# Scene classifier v2: ResNet50 fine-tuning (Team B2)

This notebook retrains the model used by the Flask app. It replaces the original training cells, which had a few bugs and whose trained model file was never published.

**How to run it on Kaggle (about 30–45 minutes on the free GPU):**
1. Kaggle → *Create* → *New Notebook* → *File* → *Import Notebook* → choose this file.
2. Right panel → *Add Input* → search **intel-image-classification** (by *puneet6060*) → *Add*.
3. Right panel → *Session options*: **Accelerator: GPU T4 x2** (or P100), **Internet: On** (needed once to download the ImageNet weights; needs a phone-verified Kaggle account).
4. *Run All*.
5. When it finishes, open the *Output* section (`/kaggle/working`) and download **`natural_scenes_image_resnet_classifier.pt`** and **`model_meta.json`**. Put both in the project folder (replace the old `.pt`) and restart the app.

**What changed from the original notebook, and why:**

| Change | Why |
|---|---|
| No vertical flips; milder rotation (±10°) | Real scene photos are never upside down; flipped mountains and seas confused glacier/mountain/sea |
| 224 px input instead of 128 px | ResNet50 was pre-trained at 224 px and works best there |
| `layer4` of ResNet is trained too (small learning rate) | Lets the network adapt its high-level features to scenes; the original froze the whole backbone |
| Validation set uses test-time transforms | The original validated on augmented images |
| Loss and accuracy averaged over every batch | The original logged only the last batch, so curves were noisy |
| Gradient clipping after `backward()` | The original clipped before `backward()`, which did nothing |
| Cosine learning-rate schedule stepped once per epoch | The original passed the loss into `StepLR`, so the rate never dropped |
| Best model (highest validation accuracy) is the one saved | The original saved the last epoch |
| Weights saved in half precision, weights-only | About 49 MB instead of 99 MB (well under GitHub's 100 MB limit) and loadable with PyTorch's safe loader |

In [ ]:
import os, json, time, copy, random, datetime

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# SMOKE_TEST=1 runs a tiny version on CPU to check the code works (used before uploading to Kaggle).
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"

def find_data_root():
    candidates = [
        os.environ.get("DATA_DIR", ""),
        "/kaggle/input/intel-image-classification",
        "./data/intel-image-classification",
        "./data",
    ]
    for c in candidates:
        if c and os.path.isdir(os.path.join(c, "seg_train", "seg_train")):
            return c
    raise FileNotFoundError(
        "Dataset not found. On Kaggle: Add Input -> search 'intel-image-classification' (puneet6060). "
        "On your own computer: set DATA_DIR to the folder that contains seg_train/ and seg_test/.")

DATA_ROOT = find_data_root()
TRAIN_DIR = os.path.join(DATA_ROOT, "seg_train", "seg_train")
TEST_DIR = os.path.join(DATA_ROOT, "seg_test", "seg_test")
EXPECTED_CLASSES = ["buildings", "forest", "glacier", "mountain", "sea", "street"]

IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 12
PATIENCE = 4            # stop if validation accuracy does not improve for this many epochs
LR_HEAD = 1e-3
LR_BACKBONE = 1e-4      # layer4 learns 10x slower than the new head
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
NUM_WORKERS = 2
SEED = 42

if SMOKE_TEST:
    IMG_SIZE, BATCH_SIZE, EPOCHS, NUM_WORKERS = 64, 8, 2, 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"   # mixed precision: faster on GPU, same accuracy

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Data:", DATA_ROOT)
print("Device:", device, "| torch", torch.__version__, "| smoke test:", SMOKE_TEST)

## Image transforms

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Must match app.py exactly: plain resize to IMG_SIZE x IMG_SIZE, then normalise.
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

## Datasets: train / validation / test

In [ ]:
train_full_aug = datasets.ImageFolder(TRAIN_DIR, transform=train_tf)
train_full_eval = datasets.ImageFolder(TRAIN_DIR, transform=eval_tf)   # same files, no augmentation
test_ds = datasets.ImageFolder(TEST_DIR, transform=eval_tf)

CLASSES = train_full_aug.classes
assert CLASSES == EXPECTED_CLASSES, f"Unexpected class folders: {CLASSES}"

g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(train_full_aug), generator=g).tolist()
n_val = int(0.2 * len(perm))
val_idx, train_idx = perm[:n_val], perm[n_val:]

train_ds = Subset(train_full_aug, train_idx)
val_ds = Subset(train_full_eval, val_idx)

loader_args = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=device.type == "cuda")
train_loader = DataLoader(train_ds, shuffle=True, drop_last=len(train_ds) > BATCH_SIZE, **loader_args)
val_loader = DataLoader(val_ds, shuffle=False, **loader_args)
test_loader = DataLoader(test_ds, shuffle=False, **loader_args)

print(f"train {len(train_ds)} | validation {len(val_ds)} | test {len(test_ds)} | classes {CLASSES}")

## Model: ResNet50 with `layer4` and a new head trained

In [ ]:
def build_model(pretrained=True):
    # Must match build_resnet() in app.py.
    weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
    net = models.resnet50(weights=weights)
    for p in net.parameters():
        p.requires_grad = False
    for p in net.layer4.parameters():
        p.requires_grad = True
    net.fc = nn.Sequential(
        nn.Linear(2048, 512), nn.ReLU(inplace=True), nn.Dropout(p=0.3),
        nn.Linear(512, 128), nn.ReLU(inplace=True), nn.Dropout(p=0.2),
        nn.Linear(128, len(EXPECTED_CLASSES)), nn.LogSoftmax(dim=1),
    )
    return net

model = build_model(pretrained=not SMOKE_TEST).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} of {total:,}")

In [ ]:
# The model outputs log-probabilities (LogSoftmax). CrossEntropyLoss applies log_softmax again,
# but log_softmax of log-probabilities changes nothing, so this equals NLLLoss and also allows label smoothing.
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW([
    {"params": model.layer4.parameters(), "lr": LR_BACKBONE},
    {"params": model.fc.parameters(), "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)

## Training

In [ ]:
def run_epoch(loader, train):
    model.train(train)
    total_loss, correct, seen = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.set_grad_enabled(train), torch.autocast(device_type=device.type, enabled=USE_AMP):
            out = model(X)
            loss = criterion(out, y)
        if train:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)                                   # clip AFTER backward
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        total_loss += loss.item() * y.size(0)                            # average over ALL batches
        correct += (out.argmax(1) == y).sum().item()
        seen += y.size(0)
    return total_loss / seen, correct / seen

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_acc, best_state, best_epoch, bad_epochs = -1.0, None, 0, 0
start = time.time()

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    scheduler.step()                                                     # once per epoch, no argument
    for k, v in zip(history, (tr_loss, tr_acc, va_loss, va_acc)):
        history[k].append(round(v, 4))
    print(f"Epoch {epoch:2d}/{EPOCHS}  train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
          f"val loss {va_loss:.4f} acc {va_acc:.4f} | {(time.time() - start) / 60:.1f} min")
    if va_acc > best_acc:
        best_acc, best_epoch, bad_epochs = va_acc, epoch, 0
        best_state = copy.deepcopy(model.state_dict())                   # keep the BEST model
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"Early stopping: no improvement for {PATIENCE} epochs.")
            break

epochs_trained = epoch
print(f"Best validation accuracy {best_acc:.4f} at epoch {best_epoch}. Total {(time.time() - start) / 60:.1f} min.")
model.load_state_dict(best_state)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="train"); axes[0].plot(history["val_loss"], label="validation")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(history["train_acc"], label="train"); axes[1].plot(history["val_acc"], label="validation")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout(); plt.show()

## Test set evaluation (photos the model never saw)

In [ ]:
model.eval()
preds, labels = [], []
with torch.no_grad():
    for X, y in test_loader:
        with torch.autocast(device_type=device.type, enabled=USE_AMP):
            out = model(X.to(device))
        preds += out.argmax(1).cpu().tolist()
        labels += y.tolist()

test_acc = float(np.mean(np.array(preds) == np.array(labels)))
report = classification_report(labels, preds, labels=list(range(len(CLASSES))), target_names=CLASSES,
                               output_dict=True, zero_division=0)
cm = confusion_matrix(labels, preds, labels=list(range(len(CLASSES))))

print(f"Test accuracy: {test_acc * 100:.2f}%\n")
print(classification_report(labels, preds, labels=list(range(len(CLASSES))), target_names=CLASSES, zero_division=0))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(xticks_rotation=45, cmap="Greens")
plt.tight_layout(); plt.show()

## Save the model and its metadata for the app

In [ ]:
MODEL_FILE = "natural_scenes_image_resnet_classifier.pt"
META_FILE = "model_meta.json"

# Weights only, in half precision (the app converts them back to float32 when loading).
state = {k: (v.detach().cpu().half() if v.is_floating_point() else v.detach().cpu()) for k, v in model.state_dict().items()}
torch.save(state, MODEL_FILE)

meta = {
    "version": "v2",
    "trained_at": datetime.date.today().isoformat(),
    "img_size": IMG_SIZE,
    "classes": CLASSES,
    "test_accuracy": round(test_acc, 4),
    "test_images": len(test_ds),
    "best_val_accuracy": round(best_acc, 4),
    "epochs_trained": epochs_trained,
    "best_epoch": best_epoch,
    "per_class": {c: {"precision": round(report[c]["precision"], 4), "recall": round(report[c]["recall"], 4),
                      "f1": round(report[c]["f1-score"], 4), "support": int(report[c]["support"])} for c in CLASSES},
    "confusion_matrix": cm.tolist(),
    "trainable": "layer4 and the classifier head fine-tuned; earlier layers frozen",
    "augmentation": "Random resized crop (60-100%), horizontal flip, rotation ±10°, brightness/contrast/saturation jitter",
    "optimizer": f"AdamW (head lr {LR_HEAD}, layer4 lr {LR_BACKBONE}), cosine schedule, label smoothing {LABEL_SMOOTHING}, mixed precision",
    "history": history,
    "smoke_test": SMOKE_TEST,
}
with open(META_FILE, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print(f"Saved {MODEL_FILE} ({os.path.getsize(MODEL_FILE) / 1e6:.1f} MB) and {META_FILE}")

## Check the saved file loads the same way the app loads it

In [ ]:
check = build_model(pretrained=False)
check.load_state_dict(torch.load(MODEL_FILE, map_location="cpu", weights_only=True))
check = check.float().eval()
X, y = next(iter(test_loader))
with torch.no_grad():
    a = check(X).argmax(1)
    b = model.float().cpu().eval()(X).argmax(1)
print(f"Reloaded model agrees with the trained model on {(a == b).float().mean().item() * 100:.1f}% of a test batch")

## Next steps
1. Download `natural_scenes_image_resnet_classifier.pt` and `model_meta.json` from the Output panel.
2. Copy both into the project folder, replacing the old `.pt` file.
3. Restart the app (`python app.py`). The *About the model* page now shows these results.
4. Commit both files on the `pranith` branch.